In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q torch tokenizers transformers tqdm


# 3. 허깅페이스 활용 — 사전 학습 모델의 지식 이식

> **📌 이 노트북의 목표**
>
> 2_2에서 만든 임베딩 층은 무작위 값으로 초기화되어 있어 의미가 없었다.
> 이 노트북에서는 **이미 잘 학습된 모델(klue/bert-base)의 임베딩 가중치를 가져와**
> 우리의 임베딩 층에 이식한다.
>
> 이것은 이후 챕터에서 배울 **전이학습(Transfer Learning)** 의 첫 번째 맛보기이다:
> "거인의 어깨에 올라타서" 처음부터 학습하지 않고도 좋은 성능을 얻는 방법.

## 3-1. 사전 준비

1. 데이터: 2_1에서 처리된 `nsmc.txt` 사용
2. 토크나이저: 2_1에서 저장한 `nsmc-vocab.txt`를 불러오기 (재학습 불필요)
3. 임베딩 레이어: 2_2에서 생성한 것과 동일 (무작위 가중치 행렬)


In [ ]:
import os
import torch
import torch.nn as nn
from tokenizers import BertWordPieceTokenizer

# ========== 2_1에서 저장한 토크나이저 불러오기 ==========
VOCAB_FILE = 'nsmc-vocab.txt'

if os.path.exists(VOCAB_FILE):
    tokenizer = BertWordPieceTokenizer(VOCAB_FILE, lowercase=False, strip_accents=False)
    print(f'저장된 토크나이저를 불러왔습니다: {VOCAB_FILE}')
else:
    print('저장된 사전이 없어 새로 학습합니다. (1~2분 소요)')
    tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
    tokenizer.train(
        files='nsmc.txt',
        vocab_size=30000,
        min_frequency=2,
        special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    )
    tokenizer.save_model('.', 'nsmc')

# 2_2에서 만든 것과 동일한 임베딩 레이어 (무작위 초기화 상태)
vocab_size = tokenizer.get_vocab_size()
embedding_dim = 768
embedding_layer = nn.Embedding(
    num_embeddings=vocab_size, embedding_dim=embedding_dim
)
print(f'준비 완료: 토크나이저(사전 {vocab_size:,}개), 임베딩({vocab_size:,}x{embedding_dim})')
print('→ 임베딩 가중치는 현재 무작위 값. 아래에서 사전학습 모델의 지식을 이식할 것.')


## 3-2. 사전 학습 모델 로드

- 허깅페이스를 통해, 한국어 데이터가 학습된 모델 불러오기

### 3-2-1. 허깅페이스에서 모델 찾기 가이드

- 허깅페이스에는 사전 훈련 모델이 매우 방대하게 공유되고 있음
- 라이브러리를 통해서 원하는 모델을 불러오기 전, 플랫폼에서 직접 원하는 모델을 검색하고 선택할 수 있어야 함.

1. 모델 검색 및 필터링
- 허깅페이스 [Models 페이지](https://huggingface.co/models)에서 검색할 때, 다양한 기준으로 검색 해 봅시다.

| 검색 기준 | 설명 | 필터링 예 |
| --- | --- | --- |
| 태스크 | 모델이 수행하는 처리 유형에 따라서 필터링 | text-classification (문장 분류), question-answering (질의응답), translation (번역) |
| 언어  | 모델이 지원하는 언어를 기준으로 필터링 | ko (한국어), en (영어), zh (중국어) 등 |
| 라이브러리 | 모델이 구현된 주요 라이브러리를 기준으로 필터링 | PyTorch, scikit, TensorFlow, JAX 등 |
| 모델 이름 | 특정 모델 이름, 저자, 또는 기술적 특징 등 | bert, gpt, google  |

### 3-2-2. 사전 학습된 모델 불러오기

- 이번 실습에서는 미리 찾아 둔, `klue/bert-base` 모델을 사용 할 것
    - 키워드: `klue`, `Transformers`, `PyTorch`, `Korean`, `bert`
    - 선정 기준
        - 토크나이징을 BERT 기반으로 하였으므로, BERT 기반의 모델
        - 임베딩을 PyTorch를 활용하고 있으므로 PyTorch 기반의 모델
        - 한국 영화 리뷰 텍스트를 처리하고 있으므로 Korean 기반의 모델

In [ ]:
# ========== 사전 학습된 모델 불러오기 ==========
# 교안 실습(2_0)에서 AutoTokenizer/AutoModel로 불러온 것과 동일한 모델이다.
# 차이점: 여기서는 모델 전체가 아닌 "임베딩 가중치"만 추출하여 사용한다.
from transformers import AutoTokenizer, AutoModel

pretrained_model_name = 'klue/bert-base'

# 사전 학습된 토크나이저와 모델 로드
# 이 모델은 대규모 한국어 데이터로 이미 학습이 완료된 상태이다.
pretrained_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)
pretrained_model = AutoModel.from_pretrained(pretrained_model_name)

# 임베딩 가중치 추출
# pretrained_model.embeddings.word_embeddings.weight:
# → 이 모델이 학습한 "단어별 의미 좌표" 행렬
# → 2_2에서 만든 embedding_layer.weight와 같은 구조이지만,
#   이쪽은 대규모 학습을 통해 의미 있는 값으로 채워져 있다.
pretrained_embeddings = pretrained_model.embeddings.word_embeddings.weight
print(f'사전학습 임베딩 크기: {pretrained_embeddings.shape}')
print(f'우리의 임베딩 크기: {embedding_layer.weight.shape}')

# 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

embedding_layer = embedding_layer.to(device)
pretrained_embeddings = pretrained_embeddings.to(device)


## 3-3. 지식 이전

무작위 가중치인 `embedding_layer`에, 잘 학습된 `pretrained_embeddings`의 내용을 옮겨 담는다.

> **💡 왜 통째로 복사하지 않는가?**
>
> 우리의 토크나이저(NSMC 데이터로 학습)와 klue/bert-base의 토크나이저는
> 단어 사전이 **완전히 같지 않다.** 사전 크기도 다르고, 같은 단어라도 ID가 다를 수 있다.
>
> 그래서 **우리 사전에 있는 단어가 사전학습 모델에도 있을 때만** 해당 벡터를 복사한다.
> 양쪽에 모두 존재하는 단어만 골라서 이식하는 것이다.


In [ ]:
from tqdm import tqdm  # 진행률 표시 바

# 우리 토크나이저의 단어 사전
custom_vocab = tokenizer.get_vocab()  # {'단어': ID, ...} 형태의 딕셔너리

# ⭐ 사전학습 토크나이저의 단어 사전도 '반복문 밖에서 한 번만' 꺼내 둔다.
#
#    AS-IS: for ...:  if token in pretrained_tokenizer.vocab:
#           → .vocab은 속성처럼 보이지만 실제로는 호출할 때마다 3만 개짜리
#             딕셔너리를 새로 만들어 낸다. 3만 번 반복하면 그 작업을 3만 번 반복하게 되어
#             몇 분씩 걸린다. (결과는 맞지만 매우 느리다)
#    TO-BE: 한 번만 꺼내서 변수에 담아두고 조회한다. 수십 초 → 1초 미만.
pretrained_vocab = pretrained_tokenizer.get_vocab()

copied_count = 0
missing_tokens = []

# torch.no_grad(): 기울기 계산을 하지 않음 (학습이 아닌 단순 복사이므로)
with torch.no_grad():
    for token, token_id in tqdm(custom_vocab.items(), desc='임베딩 이식 중'):
        # 우리 사전의 각 단어가 사전학습 모델의 사전에도 있는지 확인
        pretrained_id = pretrained_vocab.get(token)   # 없으면 None
        if pretrained_id is not None:
            # 사전학습 모델의 임베딩 벡터를 우리 임베딩 층에 복사
            # → 이 단어에 대한 "의미 좌표"가 랜덤값에서 학습된 값으로 교체된다
            embedding_layer.weight[token_id] = pretrained_embeddings[pretrained_id]
            copied_count += 1
        else:
            missing_tokens.append(token)

print(f'\n총 {len(custom_vocab):,}개 단어 중 {copied_count:,}개 벡터를 이식 완료')
print(f'이식률: {copied_count/len(custom_vocab)*100:.1f}%')
print('→ 이식되지 않은 단어는 여전히 무작위 값. 추가 학습이 필요하다.')

# 어떤 단어가 이식되지 못했는지 확인해 보자
print(f'\n이식되지 못한 토큰 예시 20개:')
print(missing_tokens[:20])
print('→ 영화 리뷰 특유의 구어체·줄임말·오타가 많다.')
print('   klue/bert-base는 뉴스·위키 위주로 학습되어 이런 표현을 모르기 때문이다.')


> **💡 이식률이 100%가 아닌 이유 — 사전이 서로 다르기 때문**
>
> ```
> 우리 사전 (영화 리뷰로 학습)        klue 사전 (뉴스·위키로 학습)
>   ┌──────────────┐                  ┌──────────────┐
>   │ ㅋㅋㅋ  존잼  │                  │ 대통령  경제  │
>   │      ┌───────┼──────────────────┼───────┐      │
>   │      │  영화  배우  감독  재미  │      │      │
>   │      └───────┼──────────────────┼───────┘      │
>   │ 개꿀  띵작   │                  │ 헌법  조례   │
>   └──────────────┘                  └──────────────┘
>          겹치는 부분만 벡터를 복사할 수 있다
> ```
>
> **실무에서는 어떻게 할까?**
>
> 1. **사전학습 모델의 토크나이저를 그대로 쓴다** ← 가장 흔하고 안전한 방법
>    - 사전이 100% 일치하므로 이식 문제가 아예 없다.
>    - 2_0에서 `AutoTokenizer`와 `AutoModel` 이름을 맞춘 이유가 바로 이것이다.
> 2. 도메인 전용 토큰만 **추가**한다 (`add_tokens`) → 추가분만 새로 학습
> 3. 데이터가 아주 많고 도메인이 특수하면 토크나이저부터 새로 만들고 전체를 학습한다
>
> 👉 이번 실습은 **"사전이 다르면 이런 문제가 생긴다"를 몸으로 겪어 보는 것**이 목적이다.


## 3-4. 결과 확인 — 지식 이식 전후 비교

2_2에서 무작위 임베딩으로 유사 단어를 찾았을 때는 의미 없는 결과가 나왔다.
사전학습 모델의 지식을 이식한 후, 같은 실험을 다시 해 보자.

> **⚠️ 기대치를 미리 조정하고 시작하자**
>
> 결과가 "완벽하게 의미적인 단어들"로 나오지 않을 수 있다. 그리고 그것이 **정상**이다.
>
> 우리가 이식한 것은 BERT의 **입력 임베딩(임베딩 층)** 이다.
> 2_0에서 봤듯 BERT가 문맥을 이해하는 능력은 그 뒤의 **Transformer 층 12개**에서 만들어진다.
> 즉 입력 임베딩만 떼어 놓고 보면, Word2Vec만큼 깔끔한 의미 관계가 나오지는 않는다.
>
> | | 무작위 (2_2) | 입력 임베딩 이식 (지금) | 문맥 임베딩 (2_0) |
> |---|---|---|---|
> | 유사도 값 | 0 근처 | 어느 정도 형성됨 | 문맥까지 반영 |
> | 유사 단어 | 완전히 무의미 | 관련어가 섞여 나옴 | 가장 정교함 |
>
> 👉 확인할 것은 "완벽한가"가 아니라 **"2_2의 무작위 결과보다 나아졌는가"** 이다.
> 유사도 수치와 단어 목록을 2_2와 나란히 비교해 보자.


In [ ]:
import torch.nn.functional as F

target_word = '영화'
top_k = 5

# 기준 단어의 ID와 벡터 조회
target_id = tokenizer.token_to_id(target_word)
if target_id is None:
    raise ValueError(f"'{target_word}'는 단어 사전에 없습니다. 다른 단어로 시도해 보세요.")

target_vector = embedding_layer.weight[target_id]

# 전체 단어 벡터와 코사인 유사도 계산
with torch.no_grad():
    all_vectors = embedding_layer.weight
    similarities = F.cosine_similarity(
        target_vector.unsqueeze(0), all_vectors, dim=1
    )

    # 유사도가 높은 Top-K 단어 찾기 (자기 자신 제외)
    top_scores, top_indices = torch.topk(similarities, k=top_k + 1)

print(f"✅ [지식 이식 후] '{target_word}'와 가장 유사한 단어 Top {top_k}:")
for i in range(1, top_k + 1):
    similar_word_id = top_indices[i].item()
    similar_word = tokenizer.id_to_token(similar_word_id)
    score = top_scores[i].item()
    print(f'  {i}순위: {similar_word} (유사도: {score:.4f})')

print('\n→ 2_2에서의 무작위 결과와 비교해 보자.')
print('   ① 유사도 수치가 0 근처에서 벗어났는가?')
print('   ② 최소한 "말이 되는" 토큰이 섞여 나오는가?')
print('→ 사전학습 모델의 "지식"이 우리의 임베딩 층에 이식된 결과다.')

print('\n💡 다른 단어로도 실험해 보자: target_word를 "배우", "재미", "감독" 등으로 바꿔 실행')
print('💡 단, 이식되지 못한 토큰을 넣으면 여전히 무작위 결과가 나온다.')
print('   (위에서 출력한 missing_tokens 목록을 참고)')


---

## 📌 챕터 2-1 전체 정리

### 우리가 따라온 길

```
"나는 개발자 입니다"        ← 사람의 문장
        ↓ 2_1 토큰화
['나는','개발','##자','입니다']   ← 의미 단위 조각
        ↓ 2_1 정수 인코딩
[2291, 12520, 1141, 2985]        ← 그냥 번호. 아직 의미 없음
        ↓ 2_2 임베딩 층
[[0.2,-1.3,...], ...]            ← 좌표. 하지만 무작위라 의미 없음
        ↓ 2_3 지식 이식
[[0.7, 0.1,...], ...]            ← 학습된 좌표. 이제 의미가 있음
        ↓ (2_0에서 미리 본 것) Transformer 층 통과
문맥까지 반영된 최종 벡터          ← 딥러닝 모델의 진짜 입력
```

### 세 번 나온 "임베딩"을 구분하자

| 이름 | 무엇 | 어디서 |
|---|---|---|
| **임베딩 층** | 정수 ID → 벡터 변환표 | 2_2에서 직접 생성, 2_3에서 이식 |
| **사전학습 임베딩** | 대규모 학습으로 채워진 표 | 2_3에서 가져온 것 |
| **문맥 임베딩** | 모델 전체를 통과한 결과 | 2_0의 `last_hidden_state` |

### 꼭 기억할 3가지

1. **토크나이저는 학습 데이터를 닮는다.** 영화 리뷰로 만든 사전은 의료 문서를 잘 못 다룬다.
2. **토크나이저와 모델은 반드시 짝을 맞춰야 한다.** 사전이 다르면 같은 번호가 다른 단어를 뜻한다.
3. **의미는 구조가 아니라 학습에서 나온다.** 같은 `(30000, 768)` 표라도 무작위면 쓸모가 없다.

### 자주 만나는 오류

| 증상 | 원인 | 해결 |
|---|---|---|
| `TypeError: ... NoneType` | `token_to_id()`가 사전에 없는 단어에 None 반환 | 존재 확인 후 사용 |
| `IndexError` (Embedding) | 토큰 ID ≥ `num_embeddings` | `tokenizer.get_vocab_size()` 기준으로 생성 |
| 이식이 몇 분씩 걸림 | 반복문 안에서 사전을 매번 새로 생성 | `get_vocab()`을 밖에서 한 번만 호출 |
| `nsmc.txt` 없음 | 2_1을 실행하지 않았거나 다른 폴더에서 실행 | 같은 디렉토리에서 2_1부터 순서대로 실행 |
